# Predicting Mouse Choice from Neural Activity

## Question

Can neural activity be used to predict whether the animal makes a left or right choice?

In this notebook, we use trial-level neural activity from the IBL Brain-Wide Map dataset and train a logistic regression classifier to predict the animal's choice.

The true choices stored in the trials data are used to evaluate the prediction accuracy.

## 1. Install Required Packages

We first install the IBL packages required to access and work with the Brain-Wide Map dataset.

`ONE-api` provides access to IBL datasets, while `ibllib` contains utilities used for working with IBL data.

In [ ]:
%pip install ONE-api --quiet
%pip install ibllib --quiet

## 2. Set Up the IBL Data Interface

The IBL data are accessed through the ONE (Open Neurophysiology Environment) interface.

Here, we configure ONE to connect to the public IBL OpenAlyx server. This interface will also provide the local cache directory where the downloaded Neuromatch dataset is stored.

In [ ]:
import os

# When running in Jupyter set number of threads to 1
os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')

from one.api import ONE

ONE.setup(base_url='https://openalyx.internationalbrainlab.org',silent=True)

one = ONE(password='international')

## Optional GPU Acceleration

The notebook runs on CPU by default.

To enable NVIDIA GPU acceleration:

1. Install RAPIDS cuML in a compatible Linux/WSL2 environment.
2. Set:

```python
USE_GPU = True

In [ ]:
# =========================
# Compute backend settings
# =========================

USE_GPU = False   # Change to True to enable NVIDIA GPU acceleration

## 3. Import Libraries

We import the libraries needed for data loading, manipulation, and classification.

The analysis mainly uses NumPy and pandas for data processing, SciPy for loading sparse neural activity matrices, and scikit-learn for training and evaluating the logistic regression classifier.

In [ ]:
import numpy as np
import pandas as pd

from scipy import sparse
from iblutil.util import Bunch

from one.remote.aws import s3_download_file

import zipfile


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import permutation_test_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt



## 4. Download Data Function

The Neuromatch tutorial provides preprocessed neural activity aligned to several behavioral events.

The function below downloads the requested dataset from the IBL storage and extracts it into the local ONE cache directory.

For this analysis, we will later use neural activity aligned to the animal's first movement (`firstMove`).

In [ ]:
def download_data(event):
    assert event in ['firstMove', 'stimOn', 'feedback'], \
        'event must be one of "firstMove", "stimOn" or "feedback"'

    # Dataset name
    fname = f'data_{event}.zip'

    # Remote location of data
    s3_data_path = f'sample_data/Neuromatch/{fname}'

    # Local location
    save_path = one.cache_dir.joinpath('Neuromatch', fname)
    save_path.parent.mkdir(exist_ok=True, parents=True)

    # Download
    file = s3_download_file(s3_data_path, save_path)

    # Unzip
    with zipfile.ZipFile(file, 'r') as zip_ref:
        zip_ref.extractall(save_path.parent)

## 5. Locate Dataset Function

After downloading the data, we need a consistent way to locate the corresponding folder in the ONE cache.

This helper function returns the local path for a dataset aligned to a specified behavioral event.

In [ ]:
def get_data_path(event):
    return one.cache_dir.joinpath('Neuromatch',f'data_{event}')

## 6. Load Metadata Function

The dataset contains several metadata tables describing the recorded neurons, trials, and recording sessions.

We also load the time vector associated with the neural activity. These timestamps specify the neural activity bins relative to the alignment event.

The resulting metadata object contains:

- cluster information,
- trial information,
- session information,
- time bins,
- sampling interval,
- and the local data path.

In [ ]:
def load_metadata(event):

    metadata = Bunch()

    data_path = get_data_path(event)

    metadata['clusters'] = pd.read_parquet(data_path.joinpath('clusters.pqt'))

    metadata['trials'] = pd.read_parquet(data_path.joinpath('trials.pqt'))

    metadata['sessions'] = pd.read_parquet(data_path.joinpath('sessions.pqt'))

    metadata['times'] = np.load(data_path.joinpath('t.npy'))

    metadata['nbins'] = metadata['times'].size

    metadata['dt'] = np.round(np.median(np.diff(metadata['times'])),2)

    metadata['data_path'] = data_path

    return metadata

## 7. Load Neural Activity Function

For each probe insertion, neural activity is stored as a sparse PSTH matrix.

The following function loads this matrix, converts it to a dense array, and reshapes it into:

**trials × neurons × time bins**

This representation allows us to examine the activity of every recorded neuron on every behavioral trial.

In [ ]:
def load_psth(data_path, pid, nbins=150):

    psth = sparse.load_npz(data_path.joinpath(f'{pid}.npz')).toarray()

    psth = psth.reshape(psth.shape[0],-1,nbins)

    return psth

## 8. Load Insertion Data Function

For the decoding analysis, neural activity and behavioral choices must correspond to the same recording session.

This helper function loads the PSTH for one probe insertion and matches it with:

- the recorded cluster information,
- the corresponding session,
- and the trial-level behavioral data.

The function returns three objects:

`psth` — trial-level neural activity  
`clust_df` — information about the recorded neurons  
`trials_df` — behavioral information for each trial

In [ ]:
def get_psth_for_insertion(pid, meta, reg=None, uuids=None):

    df = meta.clusters[meta.clusters['pid'] == pid]

    df = df[['acronym', 'pid', 'uuids', 'cluster_id', 'psth_index']]

    sp = load_psth(meta.data_path,pid,nbins=meta.nbins)

    if reg is not None:
        in_reg = df['acronym'] == reg
        sp = sp[:, in_reg.values, :]
        df = df[in_reg].reset_index(drop=True)

    if uuids is not None:
        in_uuid = df['uuids'].isin(uuids)
        sp = sp[:, in_uuid.values, :]
        df = df[in_uuid].reset_index(drop=True)

    eid = meta.sessions[meta.sessions['pid'] == pid].iloc[0]['eid']

    trials = meta.trials[meta.trials['eid'] == eid].reset_index(drop=True)

    psth = sp / meta['dt']

    return psth, df, trials

## 9. Download First-Movement Data

Our goal is to predict the animal's choice from neural activity immediately before the choice-related movement occurs.

Therefore, we use the dataset aligned to `firstMove`, where time zero corresponds to the animal's first detected movement on each trial.

In [ ]:
download_data('firstMove')

## 10. Load First-Movement Metadata

We now load the metadata associated with the `firstMove` dataset.

We also inspect the time vector and bin size to understand the temporal range of neural activity available around the movement event.

In [ ]:
meta = load_metadata('firstMove')
print(meta.times.shape)
print(meta.times[0], meta.times[-1])
print(meta.dt)

## 11. Load Example Probe Insertion

To build a simple baseline decoder, we begin with a single probe insertion.

We use the same example probe insertion used in the reference tutorial and load both its neural activity and the corresponding behavioral trials.

In [ ]:
pid = '8ca1a850-26ef-42be-8b28-c2e2d12f06d6'
psth, clust_df, trials_df = get_psth_for_insertion(pid,meta)

In [ ]:
counts_check = psth * meta['dt']

integer_error = np.max(
    np.abs(counts_check - np.round(counts_check))
)

print("Maximum deviation from integer spike counts:",
      integer_error)

## 12. Inspect Loaded Data

Before constructing the classifier, we inspect the dimensions of the neural and behavioral data.

The number of trials in the PSTH should correspond to the number of rows in the trial table, while the second PSTH dimension represents the recorded neural clusters.

In [ ]:
print("PSTH shape:", psth.shape)
print("Number of clusters:", len(clust_df))
print("Number of trials:", len(trials_df))





In [ ]:
trials_df.head()

In [ ]:
clust_df.head()

## 13. Inspect Choice Labels

The target variable for our classifier is the animal's choice on each trial.

The `choice` column in the trials table contains the behavioral choice labels. We inspect their distribution before training the classifier.

In [ ]:
trials_df['choice'].value_counts()

## 14. Isolate Pre-Movement Neural Activity

To predict the animal's upcoming choice, we focus on neural activity occurring before the movement itself.

Following the reference tutorial, we select the 100 ms period immediately preceding the first movement:

**-100 ms to 0 ms relative to first movement**

This prevents the model from using neural activity occurring after movement onset.

In [ ]:
time_mask = np.bitwise_and(meta.times < 0,meta.times > -0.1)

print(meta.times[time_mask])

## 15. Construct Neural Feature Matrix

The PSTH initially has three dimensions:

**trials × neurons × time**

Logistic regression requires a two-dimensional feature matrix:

**samples × features**

We therefore average the neural activity across the selected 100 ms time window.

After averaging, each trial is represented by one feature value per recorded neuron. Thus, every row of `X` represents one trial and every column represents one neuron.

In [ ]:
X = psth[:, :, time_mask].mean(axis=2)

print("Original PSTH:", psth.shape)
print("Model input X:", X.shape)

In [ ]:
y = trials_df.choice.values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))
assert X.shape[0] == y.shape[0]

## 16. Apply Causal Smoothing

To examine whether temporal smoothing improves decoding, we create a second version of the neural features using causal exponential smoothing.

The causal kernel uses only the current and previous time bins, preventing neural activity after the first movement from leaking into the pre-movement features.

The smoothed and unsmoothed features will later be compared using exactly the same cross-validation splits.

In [ ]:
from scipy.signal import lfilter

# Keep the original unsmoothed features
X_raw = X.copy()

# Causal exponential smoothing
tau = 0.02  # 20 ms
dt = meta['dt']

kernel_length = int(5 * tau / dt)

kernel = np.exp(
    -np.arange(kernel_length) * dt / tau
)

kernel = kernel / kernel.sum()

# Smooth along the time dimension only
psth_smooth = lfilter(
    kernel,
    [1.0],
    psth,
    axis=2
)

# Create features from the same -100 ms to 0 ms window
X_smooth = psth_smooth[:, :, time_mask].mean(axis=2)

print("Raw X shape:", X_raw.shape)
print("Smoothed X shape:", X_smooth.shape)

## 17. Check for Missing Values

Before splitting the data into training and test sets, we check whether the neural feature matrix (`X`) or the choice labels (`y`) contain missing values.

Some columns in the trial metadata, such as `contrastLeft` and `contrastRight`, contain NaN values. However, these NaNs are part of the task structure and these variables are not used as inputs to the current classifier.

The current model only uses:

- `X`: neural activity features
- `y`: the animal's left/right choice

Therefore, we only need to verify that these two variables do not contain missing values.

In [ ]:
print("NaNs in neural features (X):", np.isnan(X).sum())
print("NaNs in choice labels (y):", np.isnan(y).sum())

assert not np.isnan(X).any(), "X contains NaN values"
assert not np.isnan(y).any(), "y contains NaN values"

nan_counts = trials_df.isna().sum()

print("Columns containing NaN values:")
print(nan_counts[nan_counts > 0])

## 18. Compute Choice Modulation Index

Before training the decoder, we first ask whether individual neurons show different activity depending on the animal's upcoming choice.

For each neuron, we compare its mean activity during the 100 ms period before movement for left-choice and right-choice trials.

Following the modulation-index idea used in the reference tutorial, we define the choice modulation index as:

$$
\mathrm{Choice\ MI}
=
\frac{
\left| FR_{\mathrm{right}} - FR_{\mathrm{left}} \right|
}{
\left| FR_{\mathrm{right}} + FR_{\mathrm{left}} \right|
}
$$

where:

- $FR_{\mathrm{right}}$ is the mean pre-movement activity for right-choice trials.
- $FR_{\mathrm{left}}$ is the mean pre-movement activity for left-choice trials.

A value close to zero indicates similar activity for the two choices, whereas a larger value indicates stronger choice-related modulation.

This analysis is descriptive and is separate from the logistic-regression decoding analysis below.

In [ ]:
# Mean activity for each neuron on left- and right-choice trials
left_mean = X[y == -1].mean(axis=0)
right_mean = X[y == 1].mean(axis=0)

# Choice modulation index
denominator = np.abs(left_mean + right_mean)

choice_mi = np.divide(
    np.abs(right_mean - left_mean),
    denominator,
    out=np.zeros_like(denominator, dtype=float),
    where=denominator > 0
)

# Store results alongside cluster metadata
choice_df = clust_df.copy()

choice_df['left_mean'] = left_mean
choice_df['right_mean'] = right_mean
choice_df['choice_mi'] = choice_mi

choice_df.sort_values(
    'choice_mi',
    ascending=False
).head(10)

## 19. Test Choice Modulation Significance

A high modulation index alone does not prove that the neuron is genuinely related to choice.

To estimate how much modulation could occur by chance, we repeatedly shuffle the left/right choice labels while keeping the neural activity unchanged.

For each neuron, the observed choice MI is compared with a null distribution obtained from shuffled choice labels.

We then calculate an empirical permutation p-value and apply a false-discovery-rate (FDR) correction across neurons.

In [ ]:
rng = np.random.default_rng(0)

n_permutations = 1000
n_neurons = X.shape[1]

null_mi = np.zeros(
    (n_permutations, n_neurons)
)

for i in range(n_permutations):

    # Shuffle choice labels
    y_shuffled = rng.permutation(y)

    left_shuffled = X[y_shuffled == -1].mean(axis=0)
    right_shuffled = X[y_shuffled == 1].mean(axis=0)

    denominator = np.abs(
        left_shuffled + right_shuffled
    )

    null_mi[i] = np.divide(
        np.abs(right_shuffled - left_shuffled),
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 0
    )

In [ ]:
# Empirical permutation p-values
observed_mi = choice_df['choice_mi'].values

p_values = (
    1 +
    (null_mi >= observed_mi).sum(axis=0)
) / (n_permutations + 1)


# Benjamini-Hochberg FDR correction
def fdr_bh(p_values):

    p_values = np.asarray(p_values)

    order = np.argsort(p_values)
    ranked_p = p_values[order]

    m = len(p_values)

    q_ranked = (
        ranked_p * m /
        np.arange(1, m + 1)
    )

    q_ranked = np.minimum.accumulate(
        q_ranked[::-1]
    )[::-1]

    q_values = np.empty(m)

    q_values[order] = np.clip(
        q_ranked,
        0,
        1
    )

    return q_values


q_values = fdr_bh(p_values)

choice_df['p_perm'] = p_values
choice_df['q_fdr'] = q_values

choice_df['significant_choice_modulation'] = (
    choice_df['q_fdr'] < 0.05
)

print(
    "Significantly choice-modulated neurons:",
    choice_df['significant_choice_modulation'].sum(),
    "/",
    len(choice_df)
)

In [ ]:
choice_df[
    [
        'acronym',
        'cluster_id',
        'choice_mi',
        'p_perm',
        'q_fdr',
        'significant_choice_modulation'
    ]
].sort_values(
    'choice_mi',
    ascending=False
).head(15)

## 19.5 Choice-Related Neural Dynamics (Post-Stimulus Multi-Region PCA)
In this section, we analyze how population-level neural activity across target brain regions (PO, CA3, and SSp-tr) differentiates between Left versus Right wheel-turn choices over time following stimulus presentation. Each step in our analytical pipeline serves a specific methodological purpose:

* Post-Stimulus Time Window ($0.0\text{s}$ to $1.0\text{s}$): We restrict our analysis to the post-stimulus period to isolate the neural cascade triggered by visual presentation and track how choice selection evolves during decision formation and execution, rather than examining pre-movement baseline activity.
* Gaussian Temporal Smoothing ($\sigma=2$): Spiking data contains high-frequency Poisson noise across individual time bins. Applying a 1D Gaussian filter directly to the condition-averaged PSTHs smooths single-unit fluctuations, revealing continuous population trajectories prior to dimensionality reduction.
* Joint Principal Component Analysis (PCA): High-dimensional multi-unit recordings are difficult to visualize and interpret directly. Fitting a joint PCA across condition-averaged PSTHs projects high-dimensional firing patterns onto orthogonal axes of maximum variance, allowing us to track population trajectories in a low-dimensional state space.
* 3D & 2D Trajectory Projections: While a 2D projection ($PC1$ vs. $PC2$) offers a clear overview of primary population variance, extending to a 3D projection ($PC1, PC2, PC3$) captures key orthogonal variance that reveals non-linear divergence between choices that a flat projection might obscure.
* Continuous Euclidean Distance: To convert trajectory divergence into a quantitative metric, we compute the pointwise Euclidean distance between Left and Right state vectors across time:$$\text{Distance}(t) = \sqrt{\sum_{i=1}^{3} \left( PC_{i,\text{Left}}(t) - PC_{i,\text{Right}}(t) \right)^2}$$This reduces complex multi-dimensional state dynamics into a continuous time-series signal representing choice selectivity.
* Dual Choice Divergence Thresholds (70% & 90% Peak Thresholds): To objectively quantify when a region reliably differentiates between choices, we identify the post-stimulus timepoints where Euclidean distance reaches 70% (choice encoding onset) and 90% (peak choice commitment) of its dynamic range relative to baseline ($t=0\text{s}$).
* Comparative Regional Latencies:
1. Early baseline separation at stimulus onset ($t=0\text{s}$) yields rapid initial threshold crossings in sensory and thalamic pathways (PO: $70\% = 0.015\text{s}$, $90\% = 0.025\text{s}$).
2. High-level cognitive pathways show fast, sustained integration (CA3: $70\% = 0.055\text{s}$, $90\% = 0.065\text{s}$).
3. Somatosensory regions exhibit substantial latency delays (SSp-tr: $70\% = 0.365\text{s}$, $90\% = 0.525\text{s}$), reflecting post-decision tactile feedback during wheel manipulation. Evaluating dual thresholds alongside trajectory plots provides an objective heuristic for mapping the functional hierarchy of choice dynamics across the brain.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter1d

# Prevent interactive rendering freezes
%matplotlib inline

# --- Helper Function: Get Average PSTH for a Region and Split by Choice ---
def get_avg_psth_for_region(pid, meta, region_acronym):
    psth, clust_df, trials_df = get_psth_for_insertion(pid, meta, reg=region_acronym)

    # Filter for valid choices (-1 for left, 1 for right)
    valid_choices = trials_df['choice'].isin([-1, 1])
    filtered_psth = psth[valid_choices]
    filtered_choices = trials_df['choice'][valid_choices].values

    # Average PSTH for left and right choices (neurons, time_bins)
    avg_psth_left = filtered_psth[filtered_choices == -1].mean(axis=0)
    avg_psth_right = filtered_psth[filtered_choices == 1].mean(axis=0)

    return {'left': avg_psth_left, 'right': avg_psth_right}


# --- Setup Post-Stimulus Time Window ---
time_mask = np.bitwise_and(meta.times >= 0.0, meta.times <= 1.0)
post_stim_times = meta.times[time_mask]
SIGMA = 2  # Gaussian smoothing kernel size

# Target regions selected from high choice modulation analysis
TARGET_REGIONS = ['PO', 'CA3', 'SSp-tr']
summary_results = []

print("Starting Multi-Region Choice Divergence Analysis...\n" + "="*55)

for reg in TARGET_REGIONS:
    try:
        # Step 1: Load Choice-Split PSTHs
        avg_psths_by_choice = get_avg_psth_for_region(pid, meta, reg)
        avg_psth_left = avg_psths_by_choice['left']
        avg_psth_right = avg_psths_by_choice['right']

        # Step 2: Temporal Smoothing & Time Window Masking
        avg_psth_left_smooth = gaussian_filter1d(avg_psth_left, sigma=SIGMA, axis=1)
        avg_psth_right_smooth = gaussian_filter1d(avg_psth_right, sigma=SIGMA, axis=1)

        avg_psth_left_masked = avg_psth_left_smooth[:, time_mask]
        avg_psth_right_masked = avg_psth_right_smooth[:, time_mask]

        # Step 3: Stack for Joint PCA (2 * time_points, n_clusters)
        all_psth = np.concatenate([avg_psth_right_masked.T, avg_psth_left_masked.T], axis=0)

        # Step 4: Fit 3D PCA & Extract Trajectories
        pca = PCA(n_components=3)
        pca.fit(all_psth)

        var_ratio = pca.explained_variance_ratio_ * 100
        total_var = np.sum(var_ratio)

        traj_right = pca.transform(avg_psth_right_masked.T).T
        traj_left = pca.transform(avg_psth_left_masked.T).T

        # Step 5: Compute Euclidean Distance
        euc_dist = np.sqrt(np.sum((traj_left - traj_right) ** 2, axis=0))

        # Step 6: Compute 70% & 90% Threshold Latencies
        baseline_dist = euc_dist[0]
        peak_dist = np.max(euc_dist)

        # 70% Cutoff
        threshold_70 = baseline_dist + 0.7 * (peak_dist - baseline_dist)
        latency_idx_70 = np.argmax(euc_dist >= threshold_70)
        latency_70 = post_stim_times[latency_idx_70]

        # 90% Cutoff
        threshold_90 = baseline_dist + 0.9 * (peak_dist - baseline_dist)
        matches_90 = np.where(euc_dist >= threshold_90)[0]

        if len(matches_90) > 0:
            latency_idx_90 = matches_90[0]
            latency_90 = post_stim_times[latency_idx_90]
        else:
            latency_idx_90 = None
            latency_90 = np.nan

        # Save metrics for cross-regional table
        summary_results.append({
            'Region': reg,
            'Total Var Exp (%)': round(total_var, 2),
            'Max Distance': round(peak_dist, 3),
            'Latency 70% (s)': round(latency_70, 3),
            'Latency 90% (s)': round(latency_90, 3) if not np.isnan(latency_90) else 'N/A'
        })

        # Step 7: Print Metrics
        print(f"--- Region: {reg} ---")
        print(f"Variance Explained -> PC1: {var_ratio[0]:.2f}%, PC2: {var_ratio[1]:.2f}%, PC3: {var_ratio[2]:.2f}%")
        print(f"Total Variance (3 PCs): {total_var:.2f}% | Max Distance: {peak_dist:.3f}")
        print(f"70% Latency: {latency_70:.3f}s | 90% Latency: {latency_90:.3f}s\n")

        # Step 8: Visualizations per Region
        fig = plt.figure(figsize=(16, 4.5))

        # 3D Trajectory Plot
        ax1 = fig.add_subplot(131, projection='3d')
        ax1.plot(traj_left[0], traj_left[1], traj_left[2], color='blue', linewidth=2, label='Left Choice')
        ax1.scatter(traj_left[0, 0], traj_left[1, 0], traj_left[2, 0], color='blue', s=80, marker='o')
        ax1.plot(traj_right[0], traj_right[1], traj_right[2], color='red', linewidth=2, label='Right Choice')
        ax1.scatter(traj_right[0, 0], traj_right[1, 0], traj_right[2, 0], color='red', s=80, marker='o')
        ax1.set_xlabel('PC1')
        ax1.set_ylabel('PC2')
        ax1.set_zlabel('PC3')
        ax1.set_title(f'{reg} - 3D Trajectories\nTotal Var: {total_var:.1f}%')
        ax1.legend(loc='upper right')

        # 2D Projection Plot
        ax2 = fig.add_subplot(132)
        ax2.plot(traj_left[0], traj_left[1], color='blue', linewidth=2, label='Left Choice')
        ax2.scatter(traj_left[0, 0], traj_left[1, 0], color='blue', s=80, marker='o')
        ax2.plot(traj_right[0], traj_right[1], color='red', linewidth=2, label='Right Choice')
        ax2.scatter(traj_right[0, 0], traj_right[1, 0], color='red', s=80, marker='o')
        ax2.set_xlabel('PC1')
        ax2.set_ylabel('PC2')
        ax2.set_title(f'{reg} - 2D Projection\nPC1: {var_ratio[0]:.1f}%, PC2: {var_ratio[1]:.1f}%')
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.legend()

        # Euclidean Distance & Latency Plot
        ax3 = fig.add_subplot(133)
        ax3.plot(post_stim_times, euc_dist, color='purple', linewidth=2, label='Euclidean Distance')
        ax3.axhline(threshold_70, color='gray', linestyle='--', alpha=0.7, label='70% Threshold')
        ax3.axvline(latency_70, color='darkgray', linestyle=':', linewidth=2, label=f'70% Latency ({latency_70:.3f}s)')
        ax3.scatter(latency_70, euc_dist[latency_idx_70], color='darkgray', s=100, zorder=5)

        if not np.isnan(latency_90):
            ax3.axhline(threshold_90, color='orange', linestyle='--', alpha=0.7, label='90% Threshold')
            ax3.axvline(latency_90, color='brown', linestyle=':', linewidth=2, label=f'90% Latency ({latency_90:.3f}s)')
            ax3.scatter(latency_90, euc_dist[latency_idx_90], color='brown', s=100, zorder=5)

        ax3.set_xlabel('Time (s) post-stimulus')
        ax3.set_ylabel('Euclidean Distance')
        ax3.set_title(f'{reg} - Divergence Latency')
        ax3.grid(True, linestyle='--', alpha=0.5)
        ax3.legend()

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Skipping region '{reg}': {e}\n")

# Display Comparison Summary Table
summary_df = pd.DataFrame(summary_results)
print("="*20 + " Regional Comparison Summary " + "="*20)
display(summary_df)

## 20. Summarize Choice Modulation by Brain Region

Each recorded neuron is associated with a brain-region acronym.

We therefore group the neurons from the current probe insertion by brain region and summarize:

- the number of recorded neurons,
- the median choice modulation index,
- and the proportion of neurons that remain significant after FDR correction.

Because a single probe insertion may contain only a small number of neurons from some regions, these results should be treated as exploratory. A brain-wide analysis across many insertions will be performed later.

In [ ]:
insertion_region_summary = (
    choice_df
    .groupby('acronym')
    .agg(
        n_clusters=('choice_mi', 'size'),
        median_choice_mi=('choice_mi', 'median'),
        mean_choice_mi=('choice_mi', 'mean'),
        significant_fraction=(
            'significant_choice_modulation',
            'mean'
        )
    )
    .sort_values(
        'median_choice_mi',
        ascending=False
    )
)

insertion_region_summary

## 21. Split Data into Training and Test Sets

To evaluate whether the model can predict choices on unseen trials, we divide the data into separate training and test sets.

80% of the trials are used for training and the remaining 20% are held out for testing. The split is stratified by choice to ensure that the class balance is preserved in both sets.

The random state is fixed to make the split reproducible.

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# print("Training samples:", X_train.shape[0])
# print("Testing samples:", X_test.shape[0])

trial_indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    trial_indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Original features
X_train = X_raw[train_idx]
X_test = X_raw[test_idx]

# Smoothed features
X_train_smooth = X_smooth[train_idx]
X_test_smooth = X_smooth[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

print("Training trials:", len(train_idx))
print("Testing trials:", len(test_idx))

## 22. Cross-Validation on Training Data

Before fitting the final logistic regression model, we use cross-validation to obtain a more reliable estimate of its performance.

We use 5-fold stratified cross-validation on the training set only. In each fold, the preprocessing steps and logistic regression model are fitted using only the training portion of that fold and evaluated on the remaining validation portion.

The held-out test set is not used during cross-validation and will be used only once for the final evaluation.

In [ ]:
# Define the complete modeling pipeline
models = {

    "Logistic Regression": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('logistic', LogisticRegression(
            random_state=42,
            max_iter=5000
        ))
    ]),

    "SVM": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel='linear',
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rf', RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model_results = {}

for name, model in models.items():

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    model_results[name] = cv_scores

    print(name)
    print("Accuracy for each fold:", cv_scores)
    print(f"Mean CV accuracy: {cv_scores.mean():.3f}")
    print(f"CV standard deviation: {cv_scores.std():.3f}")
    print("-----------------------")

In [ ]:
raw_cv_results = {}

for name, model in models.items():

    raw_cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    raw_cv_results[name] = raw_cv_scores

    print(name)
    print(
        f"Raw mean CV accuracy: {raw_cv_scores.mean():.3f}"
    )
    print(
        f"Raw CV std: {raw_cv_scores.std():.3f}"
    )
    print("----------------------")

## 23. Compare Decoder to Dummy Baselines

To evaluate whether the logistic regression model consistently performs better than a simple baseline, we compare both models using the same cross-validation splits.

We use repeated stratified 5-fold cross-validation to obtain multiple performance estimates.

For each split, we calculate the difference between logistic regression accuracy and dummy-model accuracy. We then estimate a 95% confidence interval for this accuracy gain.

If the entire confidence interval is above zero, this provides evidence that the neural decoder performs consistently better than the dummy baseline.

In [ ]:
# Repeated stratified cross-validation
repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=20,
    random_state=42
)

# Save exactly the same splits for both models
cv_splits = list(
    repeated_cv.split(X_train, y_train)
)

# Model scores using identical CV splits

model_cv_scores = {}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv_splits,
        scoring='accuracy'
    )

    model_cv_scores[name] = scores

    print(f"Mean {name} CV Accuracy:",
          scores.mean())
    
# Dummy baseline
dummy_cv = DummyClassifier(
    strategy='most_frequent'
)

dummy_cv_scores = cross_val_score(
    dummy_cv,
    X_train,
    y_train,
    cv=cv_splits,
    scoring='accuracy'
)


print("Mean Dummy CV Accuracy:",
      dummy_cv_scores.mean())

In [ ]:
accuracy_gain = {}

for name, scores in model_cv_scores.items():

    accuracy_gain[name] = (
        scores -
        dummy_cv_scores
    )

    print(
        f"{name} mean accuracy gain:",
        accuracy_gain[name].mean()
    )

In [ ]:
rng = np.random.default_rng(0)

n_bootstrap = 10000

bootstrap_means = np.empty(
    n_bootstrap
)
accuracy_gain_logistic = accuracy_gain["Logistic Regression"]

for i in range(n_bootstrap):

    bootstrap_sample = rng.choice(
        accuracy_gain_logistic,
        size=len(accuracy_gain_logistic),
        replace=True
    )

    bootstrap_means[i] = (
        bootstrap_sample.mean()
    )

ci_lower, ci_upper = np.percentile(
    bootstrap_means,
    [2.5, 97.5]
)

print(
    f"Mean Accuracy Gain: "
    f"{accuracy_gain_logistic.mean():.3f}"
)

print(
    f"95% CI for Accuracy Gain: "
    f"[{ci_lower:.3f}, {ci_upper:.3f}]"
)

In [ ]:
if ci_lower > 0:

    print(
        "The logistic regression model "
        "consistently outperforms the "
        "dummy baseline."
    )

else:

    print(
        "The 95% confidence interval "
        "includes zero, so the model's "
        "improvement over the dummy "
        "baseline is not clearly established."
    )

In [ ]:
plt.figure(figsize=(7,5))

data = [
    model_cv_scores["Logistic Regression"],
    model_cv_scores["SVM"],
    model_cv_scores["Random Forest"],
    dummy_cv_scores
]

plt.boxplot(
    data,
    tick_labels=[
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "Dummy Model"
    ],
    showmeans=True
)

# Show individual CV scores
for i, values in enumerate(data, start=1):
    x = np.random.normal(
        i,
        0.04,
        size=len(values)
    )
    plt.scatter(
        x,
        values,
        alpha=0.6
    )

plt.ylabel("Cross-Validated Accuracy")
plt.title(
    "Distribution of Cross-Validated Model Accuracy"
)

plt.axhline(
    0.5,
    linestyle="--",
    label="Chance level"
)

plt.legend()
plt.show()

## 24. Visualize Confidence Interval of Accuracy Gain

To visualize whether the decoding performance is consistently better than the dummy baseline, we examine the confidence interval of the accuracy improvement across cross-validation evaluations.

In [ ]:
# Visualize accuracy improvement over dummy baseline

gain_results = {}

for name, scores in model_cv_scores.items():

    accuracy_gain = scores - dummy_cv_scores

    gain_results[name] = accuracy_gain

    gain_mean = np.mean(accuracy_gain)

    gain_ci = np.percentile(
        accuracy_gain,
        [2.5, 97.5]
    )

    print(
        f"{name} mean accuracy gain: {gain_mean:.3f}"
    )

    print(
        f"{name} 95% CI: [{gain_ci[0]:.3f}, {gain_ci[1]:.3f}]"
    )


In [ ]:
names = []
means = []
errors = []

for name, gain in gain_results.items():

    ci = np.percentile(
        gain,
        [2.5, 97.5]
    )

    names.append(name)
    means.append(gain.mean())

    errors.append([
        gain.mean() - ci[0],
        ci[1] - gain.mean()
    ])


plt.figure(figsize=(6,4))

plt.errorbar(
    names,
    means,
    yerr=np.array(errors).T,
    fmt='o',
    capsize=6
)

plt.axhline(
    0,
    linestyle="--"
)

plt.ylabel("Accuracy Improvement")
plt.title("Decoding Improvement Over Dummy Model")

plt.show()

## 25. Permutation Test for Decoding Significance

To test whether the decoding accuracy is genuinely related to the relationship between neural activity and choice, we perform a permutation test.

The true choice labels are randomly shuffled while the neural features remain unchanged. The complete cross-validation procedure is then repeated using these shuffled labels.

This creates a null distribution representing the decoding accuracy expected when there is no real relationship between neural activity and the animal's choice.

The observed cross-validated accuracy is compared with this null distribution. A small permutation p-value indicates that the decoding performance is unlikely to have occurred by chance.

In [ ]:
logistic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(
        random_state=42,
        max_iter=5000
    ))
])

observed_score, permutation_scores, permutation_pvalue = permutation_test_score(
    logistic_pipeline,
    X_train,
    y_train,
    scoring='accuracy',
    cv=cv,
    n_permutations=1000,
    n_jobs=-1,
    random_state=42
)

print(f"Observed CV Accuracy: {observed_score:.3f}")
print(f"Mean Permuted Accuracy: {permutation_scores.mean():.3f}")
print(f"Permutation p-value: {permutation_pvalue:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    permutation_scores,
    bins=30,
    alpha=0.7
)

plt.axvline(
    observed_score,
    linestyle='--',
    linewidth=2,
    label=f'Observed accuracy = {observed_score:.3f}'
)

plt.xlabel('Cross-Validated Accuracy')
plt.ylabel('Frequency')
plt.title('Permutation Test: Null Distribution of Decoding Accuracy')
plt.legend()

plt.show()

In [ ]:
alpha = 0.05

if permutation_pvalue < alpha:
    print(
        "Decoding accuracy is significantly above the shuffled-label null distribution."
    )
else:
    print(
        "Decoding accuracy is not significantly different from the shuffled-label null distribution."
    )

## 26. Train Final Logistic Regression Classifier

We now train a logistic regression classifier using the neural activity from the training trials.

The model learns a weight for each neural feature and uses the resulting combination of neural activity to distinguish between the two behavioral choices.

In [ ]:
clf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(
        random_state=42,
        max_iter=5000
    ))
])

# Train final model using all training trials
clf.fit(X_train, y_train)

## 27. Train Dummy Baseline Models

Before evaluating the logistic regression classifier, we establish simple baseline models that do not use the neural activity in a meaningful way.

These baselines help determine whether the neural decoder performs better than trivial prediction strategies.

We use two dummy classifiers:

- **Most Frequent:** always predicts the most common choice observed in the training set.
- **Stratified Random:** generates predictions randomly while approximately preserving the class distribution observed in the training data.

A useful neural decoder should perform substantially better than these baselines.

In [ ]:
dummy_most_frequent = DummyClassifier(strategy='most_frequent')

dummy_stratified = DummyClassifier(strategy='stratified',random_state=0)

dummy_most_frequent.fit(X_train, y_train)
dummy_stratified.fit(X_train, y_train)

## 28. Inspect Logistic Regression Feature Weights

The logistic regression model assigns a weight to each neural feature. Larger absolute weights indicate stronger contribution of a neuron to predicting the animal's choice.

In [ ]:
# Extract logistic regression coefficients

logistic_weights = clf.named_steps['logistic'].coef_[0]

# Rank neurons by absolute contribution

top_neurons = np.argsort(
    np.abs(logistic_weights)
)[::-1]

top_weight_table = pd.DataFrame({
    "neuron_index": top_neurons[:20],
    "logistic_weight": logistic_weights[top_neurons[:20]]
})

top_weight_table

print("Top contributing neurons:")
print(top_neurons[:20])

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    logistic_weights,
    bins=50
)

plt.xlabel("Logistic Regression Weight")
plt.ylabel("Number of Neurons")
plt.title(
    "Distribution of Neural Feature Contributions"
)

plt.show()

## 29. Predict Choices on Held-Out Test Data

After training, we apply the classifier to the held-out test trials.

The predicted labels can then be directly compared with the animal's true choices stored in `y_test`.

In [ ]:
choice_predicted = clf.predict(X_test)

dummy_most_predicted = dummy_most_frequent.predict(X_test)

dummy_stratified_predicted = dummy_stratified.predict(X_test)

print("True choices:")
print(y_test[:20])

print()

print("Predicted choices:")
print(choice_predicted[:20])

## 30. Confusion Matrix of Decoder Predictions

The confusion matrix shows how well the decoder distinguishes between left and right choices on unseen trials.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test,
    choice_predicted
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Left", "Right"]
)

disp.plot()

plt.title("Confusion Matrix - Logistic Regression Decoder")
plt.show()

## 31. Evaluate Decoder Against Dummy Baselines

To determine whether the neural activity contains meaningful information about the animal's choice, we compare the logistic regression classifier with the dummy baseline models.

All models are evaluated on exactly the same held-out test trials.

If logistic regression achieves higher accuracy than the dummy classifiers, this suggests that the neural population activity contains predictive information about the animal's upcoming choice.

In [ ]:
logistic_accuracy = accuracy_score(y_test,choice_predicted)

most_frequent_accuracy = accuracy_score(y_test,dummy_most_predicted)

stratified_accuracy = accuracy_score(y_test,dummy_stratified_predicted)

print(f"Logistic Regression Accuracy: {logistic_accuracy:.2%}")
print(f"Most Frequent Dummy Accuracy: {most_frequent_accuracy:.2%}")
print(f"Stratified Dummy Accuracy: {stratified_accuracy:.2%}")

## 32. Visualize Test Accuracy Comparison

The performance of the neural decoder is compared with simple baseline models.

In [ ]:
plt.figure(figsize=(6,4))

models = [
    "Logistic Regression",
    "Most Frequent Dummy",
    "Stratified Dummy"
]

accuracies = [
    logistic_accuracy,
    most_frequent_accuracy,
    stratified_accuracy
]

plt.bar(
    models,
    accuracies
)

plt.ylabel("Accuracy")
plt.title(
    "Decoder Performance Compared with Dummy Baselines"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.ylim(0,1)

plt.show()

## 33. Conclusion: Single Probe Insertion

In this analysis, neural activity recorded immediately before the animal's first movement was used to predict its left/right choice.

Each trial was represented by the average activity of the recorded neurons during the 100 ms period preceding movement onset. A logistic regression classifier was then trained on a subset of trials and evaluated on held-out trials.

The resulting test accuracy provides a simple measure of how much information about the animal's upcoming choice is present in the recorded neural population activity.

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Most Frequent Dummy',
        'Stratified Dummy'
    ],
    'Accuracy': [
        logistic_accuracy,
        most_frequent_accuracy,
        stratified_accuracy
    ]
})

results

## Extension A: Brain-Wide Choice Modulation Analysis

The previous analysis examined choice modulation within a single probe insertion.

We now extend the same idea across the Brain-Wide Map dataset.

For every probe insertion, we calculate the choice modulation index of each neuron using neural activity during the 100 ms period preceding first movement.

Each neuron's modulation is calculated within its own recording session before neurons are pooled by anatomical brain region.

This avoids directly combining trials from different animals or sessions.

In [ ]:
choice_mi_results = []

pids = meta.clusters['pid'].unique()

for pid_i in pids:

    psth_i, clust_i, trials_i = get_psth_for_insertion(
        pid_i,
        meta
    )

    # Same pre-movement feature window used by the decoder
    X_i = psth_i[:, :, time_mask].mean(axis=2)

    y_i = trials_i.choice.values

    # Keep only valid left/right choices
    valid_trials = np.isin(y_i, [-1, 1])

    X_i = X_i[valid_trials]
    y_i = y_i[valid_trials]

    # Need both choices to calculate choice modulation
    if len(np.unique(y_i)) < 2:
        continue

    left_mean_i = X_i[y_i == -1].mean(axis=0)
    right_mean_i = X_i[y_i == 1].mean(axis=0)

    denominator = np.abs(
        left_mean_i + right_mean_i
    )

    mi_i = np.divide(
        np.abs(right_mean_i - left_mean_i),
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 0
    )

    temp = clust_i.copy()

    temp['choice_mi'] = mi_i

    choice_mi_results.append(temp)


all_choice_mi = pd.concat(
    choice_mi_results,
    ignore_index=True
)

print(
    "Total neurons analyzed:",
    len(all_choice_mi)
)

### A.1. Brain-Region Summary

Following the general brain-wide analysis strategy used in the reference tutorial, we now summarize neural modulation by anatomical region.

For each region, we calculate:

- the total number of recorded neurons,
- the median choice MI,
- the mean choice MI,
- and the fraction of neurons with a relatively high choice MI.

The MI threshold of 0.3 is used here only as a descriptive heuristic, similar to the tutorial. It should not be interpreted as statistical significance.

In [ ]:
mi_threshold = 0.3

all_choice_mi['high_choice_mi'] = (
    (all_choice_mi['choice_mi'] >= mi_threshold) &
    (all_choice_mi['choice_mi'] < 1)
)

brain_region_summary = (
    all_choice_mi
    .groupby('acronym')
    .agg(
        n_clusters=('choice_mi', 'size'),
        median_choice_mi=('choice_mi', 'median'),
        mean_choice_mi=('choice_mi', 'mean'),
        high_mi_fraction=('high_choice_mi', 'mean')
    )
)

brain_region_summary.head()

### A.2. Restrict to Well-Sampled Brain Regions

Regions represented by only a small number of neurons can show unstable or misleading modulation estimates.

Following the reference tutorial, we restrict the descriptive brain-wide comparison to regions containing at least 200 recorded clusters.

In [ ]:
min_clusters = 200

brain_region_filtered = (
    brain_region_summary[
        brain_region_summary['n_clusters'] >= min_clusters
    ]
    .sort_values(
        'high_mi_fraction',
        ascending=False
    )
)

brain_region_filtered.head(20)

In [ ]:
top_regions = brain_region_filtered.head(20)

ax = top_regions['high_mi_fraction'].plot(
    kind='bar',
    figsize=(12, 5)
)

ax.set_xlabel('Brain region')
ax.set_ylabel('Fraction of neurons with Choice MI >= 0.3')
ax.set_title('Brain-wide Choice Modulation')

plt.tight_layout()
plt.show()

## Extension B: Region-Wise Choice Decoding

Choice modulation measures whether neural activity differs between left and right choices, but the main question of this project is predictive:

**Can neural activity predict the animal's upcoming choice?**

We therefore train separate logistic-regression decoders for individual brain regions.

Importantly, decoding is performed separately within each probe insertion. We do not combine neurons from different sessions into a single feature matrix.

For each region-insertion pair, the logistic-regression accuracy is compared with a most-frequent dummy classifier evaluated on exactly the same test trials.

In [ ]:
region_decoding_results = []

pids = meta.clusters['pid'].unique()

for pid_i in pids:

    psth_i, clust_i, trials_i = get_psth_for_insertion(
        pid_i,
        meta
    )

    # Reset cluster indices so they match the PSTH neuron axis
    clust_i = clust_i.reset_index(drop=True)

    # Sanity check: one metadata row per PSTH neuron
    assert psth_i.shape[1] == len(clust_i)

    y_i = trials_i.choice.values

    valid_trials = np.isin(y_i, [-1, 1])

    y_valid = y_i[valid_trials]

    # Skip sessions without both classes
    if len(np.unique(y_valid)) < 2:
        continue

    # Analyze each region recorded by this insertion
    for reg in clust_i['acronym'].dropna().unique():

        if reg in ['root', 'void']:
            continue

        # Positional indices of neurons belonging to this region
        cluster_indices = np.flatnonzero(
            clust_i['acronym'].to_numpy() == reg
        )

        if len(cluster_indices) < 5:
            continue

        # Require at least a few neurons
        if len(cluster_indices) < 5:
            continue

        X_reg = (
            psth_i[
                :,
                cluster_indices,
                :
            ][:, :, time_mask]
            .mean(axis=2)
        )

        X_reg = X_reg[valid_trials]

        X_train_reg, X_test_reg, y_train_reg, y_test_reg = (
            train_test_split(
                X_reg,
                y_valid,
                test_size=0.5,
                random_state=42,
                stratify=y_valid
            )
        )

        # Logistic regression
        clf_reg = LogisticRegression(
            random_state=42,
            max_iter=1000
        )

        # Scale neural features using training data only
        scaler_reg = StandardScaler()

        X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
        X_test_reg_scaled = scaler_reg.transform(X_test_reg)

        # Logistic regression
        clf_reg = LogisticRegression(
            random_state=42,
            max_iter=5000
        )

        clf_reg.fit(
            X_train_reg_scaled,
            y_train_reg
        )

        pred_reg = clf_reg.predict(
            X_test_reg_scaled
        )

        logistic_acc = accuracy_score(
            y_test_reg,
            pred_reg
        )

        # Dummy baseline
        dummy_reg = DummyClassifier(
            strategy='most_frequent'
        )

        dummy_reg.fit(
            X_train_reg,
            y_train_reg
        )

        dummy_pred = dummy_reg.predict(
            X_test_reg
        )

        dummy_acc = accuracy_score(
            y_test_reg,
            dummy_pred
        )

        region_decoding_results.append({
            'pid': pid_i,
            'acronym': reg,
            'n_clusters': len(cluster_indices),
            'logistic_accuracy': logistic_acc,
            'dummy_accuracy': dummy_acc,
            'accuracy_gain': logistic_acc - dummy_acc
        })

In [ ]:
region_decoding_results = pd.DataFrame(
    region_decoding_results
)

region_decoding_results.head()

### B.1. Aggregate Decoding Performance Across Insertions

A brain region may be recorded in multiple independent probe insertions.

Rather than combining these recordings into one artificial neural population, we summarize the decoding performance obtained independently from each insertion.

For each region, we report the median logistic-regression accuracy, the median dummy accuracy, and the median improvement over the dummy baseline.

In [ ]:
region_decoding_summary = (
    region_decoding_results
    .groupby('acronym')
    .agg(
        n_insertions=('pid', 'nunique'),

        median_accuracy=(
            'logistic_accuracy',
            'median'
        ),

        median_dummy_accuracy=(
            'dummy_accuracy',
            'median'
        ),

        median_accuracy_gain=(
            'accuracy_gain',
            'median'
        ),

        median_n_clusters=(
            'n_clusters',
            'median'
        )
    )
)

In [ ]:
region_decoding_filtered = (
    region_decoding_summary[
        region_decoding_summary['n_insertions'] >= 3
    ]
    .sort_values(
        'median_accuracy_gain',
        ascending=False
    )
)

region_decoding_filtered.head(20)

## Extension C: Combine Choice Modulation and Decoding Results

The two analyses provide complementary information:

- **Choice MI** measures how strongly neural activity differs between left and right choices.
- **Decoding accuracy** measures how well neural population activity predicts choice on held-out trials.

We therefore combine the region-level results to identify brain regions that show both strong choice-related modulation and useful predictive information.

In [ ]:
combined_region_results = (
    brain_region_filtered
    .join(
        region_decoding_filtered,
        how='inner'
    )
)

combined_region_results = (
    combined_region_results
    .sort_values(
        'median_accuracy_gain',
        ascending=False
    )
)

combined_region_results[
    [
        'n_clusters',
        'median_choice_mi',
        'high_mi_fraction',
        'n_insertions',
        'median_accuracy',
        'median_dummy_accuracy',
        'median_accuracy_gain'
    ]
].head(20)

## Extension D: Leave-One-Region-Out Ablation Analysis

To estimate how much each brain region contributes to choice decoding, we perform a leave-one-region-out ablation analysis.

For each recording insertion, we first train and evaluate a decoder using all available neurons. We then remove the neurons belonging to one brain region at a time and repeat the decoding analysis using exactly the same cross-validation splits.

The decrease in cross-validated accuracy after removing a region is used as a measure of its contribution to choice decoding.

A larger positive accuracy drop indicates that removing that region causes a stronger loss of choice-related information.

Because this is a computational ablation rather than a biological intervention, the results indicate contribution to decoding and should not be interpreted as evidence of a causal effect on behavior.

In [ ]:
ablation_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(
        random_state=42,
        max_iter=5000
    ))
])

ablation_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

min_clusters_per_region = 5

In [ ]:
ablation_results = []

pids = meta.clusters['pid'].dropna().unique()

for pid_i in pids:

    try:
        # Load one insertion
        psth_i, clust_i, trials_i = get_psth_for_insertion(
            pid_i,
            meta
        )

        # Important: make cluster indices positional
        clust_i = clust_i.reset_index(drop=True)

        assert psth_i.shape[1] == len(clust_i)

        # Choice labels
        y_i = trials_i.choice.values

        # Keep only valid left/right choices
        valid_trials = np.isin(y_i, [-1, 1])

        y_valid = y_i[valid_trials]

        # Need both classes
        classes, counts = np.unique(
            y_valid,
            return_counts=True
        )

        if len(classes) < 2:
            continue

        # Need enough trials for 5-fold CV
        if counts.min() < 5:
            continue

        # Neural features:
        # mean activity during -100 ms to first movement
        X_all = (
            psth_i[:, :, time_mask]
            .mean(axis=2)
        )

        X_all = X_all[valid_trials]

        # Exactly the same CV splits will be used
        # for the full and ablated models
        cv_splits = list(
            ablation_cv.split(
                X_all,
                y_valid
            )
        )

        # Full model
        full_scores = cross_val_score(
            ablation_pipeline,
            X_all,
            y_valid,
            cv=cv_splits,
            scoring='accuracy'
        )

        full_accuracy = full_scores.mean()

        # Brain regions in this insertion
        regions = (
            clust_i['acronym']
            .dropna()
            .unique()
        )

        for region in regions:

            if region in ['root', 'void']:
                continue

            region_mask = (
                clust_i['acronym']
                .to_numpy()
                == region
            )

            n_removed = region_mask.sum()

            # Ignore regions represented by very few neurons
            if n_removed < min_clusters_per_region:
                continue

            keep_mask = ~region_mask

            # Cannot decode if nothing remains
            if keep_mask.sum() == 0:
                continue

            # Remove this region
            X_without_region = X_all[:, keep_mask]

            # Same trials + same CV folds
            ablated_scores = cross_val_score(
                ablation_pipeline,
                X_without_region,
                y_valid,
                cv=cv_splits,
                scoring='accuracy'
            )

            ablated_accuracy = (
                ablated_scores.mean()
            )

            accuracy_drop = (
                full_accuracy
                - ablated_accuracy
            )

            ablation_results.append({
                'pid': pid_i,
                'acronym': region,
                'n_removed_clusters': n_removed,
                'n_total_clusters': X_all.shape[1],
                'full_accuracy': full_accuracy,
                'ablated_accuracy': ablated_accuracy,
                'accuracy_drop': accuracy_drop
            })

    except Exception as e:

        print(
            f"Skipping {pid_i}: {e}"
        )

In [ ]:
ablation_results = pd.DataFrame(
    ablation_results
)

print(
    "Number of region-insertion comparisons:",
    len(ablation_results)
)

ablation_results.head()

### D.1. Region-Level Ablation Summary

Because the same brain region may appear in multiple recording insertions, we summarize the ablation effect across insertions.

Regions with a consistently larger positive accuracy drop are interpreted as carrying more unique information useful for choice decoding.

In [ ]:
region_ablation_summary = (
    ablation_results
    .groupby('acronym')
    .agg(
        n_insertions=(
            'pid',
            'nunique'
        ),
        median_accuracy_drop=(
            'accuracy_drop',
            'median'
        ),
        mean_accuracy_drop=(
            'accuracy_drop',
            'mean'
        ),
        median_full_accuracy=(
            'full_accuracy',
            'median'
        ),
        median_ablated_accuracy=(
            'ablated_accuracy',
            'median'
        ),
        median_removed_clusters=(
            'n_removed_clusters',
            'median'
        )
    )
    .reset_index()
)

In [ ]:
region_ablation_summary = (
    ablation_results
    .groupby('acronym')
    .agg(
        n_insertions=('pid', 'nunique'),
        median_accuracy_drop=('accuracy_drop', 'median'),
        mean_accuracy_drop=('accuracy_drop', 'mean'),
        median_full_accuracy=('full_accuracy', 'median'),
        median_ablated_accuracy=('ablated_accuracy', 'median'),
        median_removed_clusters=('n_removed_clusters', 'median')
    )
    .reset_index()
)

region_ablation_summary.head()

In [ ]:
region_ablation_filtered = (
    region_ablation_summary[
        region_ablation_summary['n_insertions'] >= 3
    ]
    .sort_values(
        'median_accuracy_drop',
        ascending=False
    )
)

region_ablation_filtered.head(20)

In [ ]:
top_regions = (
    region_ablation_filtered
    .head(15)
    .sort_values(
        'median_accuracy_drop'
    )
)

plt.figure(figsize=(8, 6))

plt.barh(
    top_regions['acronym'],
    top_regions['median_accuracy_drop']
)

plt.axvline(
    0,
    linestyle='--'
)

plt.xlabel(
    'Median Drop in Cross-Validated Accuracy'
)

plt.ylabel(
    'Brain Region'
)

plt.title(
    'Leave-One-Region-Out Ablation Analysis'
)

plt.tight_layout()
plt.show()

### D.2. Neuron-Count-Matched Random Ablation

To test whether the observed effect of removing a brain region can be explained simply by the number of neurons removed, we compare each regional ablation with matched random ablations.

For each region, we randomly remove the same number of neurons from the other neurons recorded in the same insertion and repeat the decoding analysis using the same cross-validation splits.

This procedure generates a null distribution of accuracy drops expected from removing the same number of neurons without specifically removing the target region.

A region is considered to show stronger-than-expected decoding importance when its observed accuracy drop is larger than the matched random-ablation distribution.

Permutation p-values are corrected across brain regions using FDR.

In [ ]:
from tqdm.auto import tqdm

rng = np.random.default_rng(0)

# Use a small number while testing the pipeline
n_random_ablations = 20

matched_ablation_results = []

# Only analyze regions that appeared in at least
# 3 insertions in D.1
valid_regions = set(
    region_ablation_filtered['acronym']
)

print("Number of regions included:", len(valid_regions))
print("Regions:", sorted(valid_regions))

In [ ]:
for pid_i in tqdm(
    ablation_results['pid'].unique(),
    desc='Processing insertions'
):

    try:
        # Load insertion
        psth_i, clust_i, trials_i = get_psth_for_insertion(
            pid_i,
            meta
        )

        clust_i = clust_i.reset_index(drop=True)

        assert psth_i.shape[1] == len(clust_i)

        # Choice labels
        y_i = trials_i.choice.values

        valid_trials = np.isin(
            y_i,
            [-1, 1]
        )

        y_valid = y_i[valid_trials]

        classes, counts = np.unique(
            y_valid,
            return_counts=True
        )

        if len(classes) < 2:
            continue

        if counts.min() < 5:
            continue

        # Neural features
        X_all = (
            psth_i[:, :, time_mask]
            .mean(axis=2)
        )

        X_all = X_all[valid_trials]

        # Same CV splits for every comparison
        cv_splits = list(
            ablation_cv.split(
                X_all,
                y_valid
            )
        )

        # Get regions already analyzed in D.1
        pid_regions = ablation_results[
            ablation_results['pid'] == pid_i
        ]

        for _, row in pid_regions.iterrows():

            region = row['acronym']

            # Skip regions with insufficient insertion coverage
            if region not in valid_regions:
                continue

            observed_drop = row[
                'accuracy_drop'
            ]

            region_mask = (
                clust_i['acronym']
                .to_numpy()
                == region
            )

            region_indices = np.flatnonzero(
                region_mask
            )

            other_indices = np.flatnonzero(
                ~region_mask
            )

            n_removed = len(
                region_indices
            )

            # Need enough non-target neurons
            if len(other_indices) < n_removed:
                continue

            random_drops = []

            for r in range(
                n_random_ablations
            ):

                # Randomly select the same number
                # of neurons from outside the target region
                random_removed = rng.choice(
                    other_indices,
                    size=n_removed,
                    replace=False
                )

                keep_mask = np.ones(
                    X_all.shape[1],
                    dtype=bool
                )

                keep_mask[
                    random_removed
                ] = False

                X_random_ablation = (
                    X_all[:, keep_mask]
                )

                random_scores = (
                    cross_val_score(
                        ablation_pipeline,
                        X_random_ablation,
                        y_valid,
                        cv=cv_splits,
                        scoring='accuracy',
                        n_jobs=-1
                    )
                )

                random_accuracy = (
                    random_scores.mean()
                )

                random_drop = (
                    row['full_accuracy']
                    - random_accuracy
                )

                random_drops.append(
                    random_drop
                )

            random_drops = np.asarray(
                random_drops
            )

            # One-sided permutation p-value
            p_value = (
                1
                + np.sum(
                    random_drops
                    >= observed_drop
                )
            ) / (
                n_random_ablations
                + 1
            )

            matched_ablation_results.append({
                'pid': pid_i,
                'acronym': region,
                'random_drops': random_drops,
                'n_removed_clusters': n_removed,
                'observed_drop': observed_drop,
                'mean_random_drop': random_drops.mean(),
                'median_random_drop': np.median(random_drops),
                'random_95_low': np.percentile(
                    random_drops,
                    2.5
                ),
                'random_95_high': np.percentile(
                    random_drops,
                    97.5
                ),
                'p_matched': p_value
            })

    except Exception as e:

        print(
            f"Skipping {pid_i}: {e}"
        )

In [ ]:
matched_ablation_results = pd.DataFrame(
    matched_ablation_results
)

print(
    "Number of matched ablation comparisons:",
    len(matched_ablation_results)
)

matched_ablation_results.head()

### D.3. Region-Level Matched Ablation Summary

For each brain region, we summarize the observed and neuron-count-matched random ablation effects across recording insertions.

The observed median accuracy drop is compared with the distribution expected from removing the same number of randomly selected neurons.

In [ ]:
matched_region_summary = (
    matched_ablation_results
    .groupby('acronym')
    .agg(
        n_insertions=(
            'pid',
            'nunique'
        ),
        median_observed_drop=(
            'observed_drop',
            'median'
        ),
        median_random_drop=(
            'median_random_drop',
            'median'
        ),
        median_p_value=(
            'p_matched',
            'median'
        )
    )
    .reset_index()
)

matched_region_summary = (
    matched_region_summary[
        matched_region_summary[
            'n_insertions'
        ] >= 3
    ]
)

matched_region_summary = (
    matched_region_summary
    .sort_values(
        'median_observed_drop',
        ascending=False
    )
)

matched_region_summary.head(20)

In [ ]:
region_matched_stats = []

for region, df_region in matched_ablation_results.groupby(
    'acronym'
):

    if df_region['pid'].nunique() < 3:
        continue

    observed_region_drop = np.median(
        df_region['observed_drop']
    )

    null_region_drops = []

    for r in range(
        n_random_ablations
    ):

        random_values = [
            drops[r]
            for drops in df_region[
                'random_drops'
            ]
        ]

        null_region_drops.append(
            np.median(
                random_values
            )
        )

    null_region_drops = np.asarray(
        null_region_drops
    )

    p_region = (
        1
        + np.sum(
            null_region_drops
            >= observed_region_drop
        )
    ) / (
        n_random_ablations
        + 1
    )

    region_matched_stats.append({
        'acronym': region,
        'n_insertions': df_region[
            'pid'
        ].nunique(),
        'observed_median_drop':
            observed_region_drop,
        'null_median_drop':
            np.median(
                null_region_drops
            ),
        'null_95_low':
            np.percentile(
                null_region_drops,
                2.5
            ),
        'null_95_high':
            np.percentile(
                null_region_drops,
                97.5
            ),
        'p_value': p_region
    })

region_matched_stats = pd.DataFrame(
    region_matched_stats
)

In [ ]:
region_matched_stats['q_fdr'] = fdr_bh(
    region_matched_stats['p_value'].values
)

region_matched_stats[
    'significant'
] = (
    region_matched_stats[
        'q_fdr'
    ] < 0.05
)

region_matched_stats = (
    region_matched_stats
    .sort_values(
        'observed_median_drop',
        ascending=False
    )
)

region_matched_stats.head(20)

In [ ]:
plot_df = (
    region_matched_stats
    .head(15)
    .sort_values(
        'observed_median_drop'
    )
)

plt.figure(figsize=(9, 7))

plt.barh(
    plot_df['acronym'],
    plot_df['observed_median_drop'],
    alpha=0.8,
    label='Observed regional ablation'
)

plt.scatter(
    plot_df['null_median_drop'],
    plot_df['acronym'],
    marker='o',
    label='Matched random ablation'
)

plt.xlabel(
    'Median Drop in Cross-Validated Accuracy'
)

plt.ylabel(
    'Brain Region'
)

plt.title(
    'Region Ablation vs Neuron-Count-Matched Random Ablation'
)

plt.legend()

plt.tight_layout()

plt.show()